In [ ]:
import os

# Colle ton token ici (celui sur l'écran)
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_281b1a89e087f070531afb856fc923d6'  # ← ton token

# Ensuite télécharge
import kagglehub
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
print("Path:", path)
# Voir les fichiers disponibles


100%|██████████| 199k/199k [00:00<00:00, 53.1MB/s]

Extracting files...
Path: /root/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques


In [9]:
import os
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(os.listdir(path))
# Charger les données
train_df = pd.read_csv(f"{path}/train.csv")
test_df  = pd.read_csv(f"{path}/test.csv")
print(f"Train : {train_df.shape}")   # (1460, 81)
print(f"Test  : {test_df.shape}")    # (1459, 80)
train_df.head()

['train.csv', 'sample_submission.csv', 'data_description.txt', 'test.csv']
Train : (1460, 81)
Test  : (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [15]:
# Séparer la cible AVANT de traiter les colonnes numériques
y = train_df['SalePrice'].copy()
train_df_feat = train_df.drop(columns=['SalePrice'])

# Colonnes numériques communes aux deux DataFrames
num_cols = train_df_feat.select_dtypes(include='number').columns

for col in num_cols:
    median = train_df_feat[col].median()
    train_df_feat[col].fillna(median, inplace=True)
    if col in test_df.columns:          # ← vérifie que la colonne existe dans test
        test_df[col].fillna(median, inplace=True)

# Colonnes catégorielles
cat_cols = train_df_feat.select_dtypes(include='object').columns
for col in cat_cols:
    mode = train_df_feat[col].mode()[0]
    train_df_feat[col].fillna(mode, inplace=True)
    if col in test_df.columns:
        test_df[col].fillna(mode, inplace=True)

print("Manquants train:", train_df_feat.isnull().sum().sum())  # 0
print("Manquants test :", test_df.isnull().sum().sum())        # 0


Manquants train: 0
Manquants test : 0


/tmp/ipykernel_67536/3471114784.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df_feat[col].fillna(median, inplace=True)
/tmp/ipykernel_67536/3471114784.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', t

In [13]:
train_enc = pd.get_dummies(train_df_feat)
test_enc  = pd.get_dummies(test_df)

# Aligner les colonnes
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

print(f"Train : {train_enc.shape} | Test : {test_enc.shape}")


Train : (1460, 286) | Test : (1459, 286)


##### preparation des donnees





In [23]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- 1. Standardisation ---
# Les réseaux de neurones détestent les grandes valeurs et les échelles différentes !
# GrLivArea est en milliers (1500 sqft) mais OverallQual est (entre 1 et 10).
# On standardise donc toutes nos données autour de 0.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_enc)
test_scaled = scaler.transform(test_enc)

# De même, il est TRÈS utile de passer la cible (le prix) en Logarithme
# pour éviter que les maisons à 500k dominent l'entraînement
import numpy as np
y_log = np.log1p(y)

# --- 2. Train / Validation Split ---
# On coupe le set d'entraînement pour avoir un set de validation local
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_log, test_size=0.2, random_state=42)

# --- 3. Conversion en Tenseurs PyTorch ---
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

# Le test final pour Kaggle
X_test_final = torch.tensor(test_scaled, dtype=torch.float32)

# --- 4. DataLoaders ---
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)


In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy




class Mlp(nn.Module):
    def __init__(self, n_input, n_hidden, n_out, dropout):
        super().__init__()
        self.w1 = nn.Linear(n_input, n_hidden)
        self.bn1 = nn.BatchNorm1d(n_hidden)
        self.w2 = nn.Linear(n_hidden, n_hidden)
        self.bn2 = nn.BatchNorm1d(n_hidden)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(n_hidden, n_out)
    # REGARDE ICI: Cet espace à gauche est obligatoire !
    def forward(self, x): 
        x = self.drop(F.relu(self.bn1(self.w1(x))))
        x = self.drop(F.relu(self.bn2(self.w2(x))))
        return self.out(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialisation du modèle
n_features = X_train_t.shape[1]
model = Mlp(n_input=n_features, n_hidden=256, n_out=1, dropout=0.3).to(device)

# Pour la régression sur le prix log-scale, on utilise MSELoss
criterion = nn.MSELoss() 

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 150
best_val_loss = float('inf')
best_model_weights = None

for epoch in range(epochs):
    # ==== ENTRAÎNEMENT ====
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # ==== VALIDATION ====
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            preds = model(batch_X)
            loss = criterion(preds, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            
    val_loss = val_loss / len(val_loader.dataset)
    
    # RMSE (Root Mean Squared Error) car c'est la métrique de Kaggle !
    rmse_train = np.sqrt(train_loss)
    rmse_val = np.sqrt(val_loss)
    
    # Sauvegarde du meilleur modèle
    if rmse_val < best_val_loss:
        best_val_loss = rmse_val
        best_model_weights = copy.deepcopy(model.state_dict())
        
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Train RMSE: {rmse_train:.4f} | Val RMSE: {rmse_val:.4f}")

print(f"\n✅ Meilleur Val RMSE : {best_val_loss:.4f}")
model.load_state_dict(best_model_weights)


Epoch  10/150 | Train RMSE: 1.1598 | Val RMSE: 1.4621
Epoch  20/150 | Train RMSE: 1.1279 | Val RMSE: 1.2919
Epoch  30/150 | Train RMSE: 1.0930 | Val RMSE: 1.2259
Epoch  40/150 | Train RMSE: 1.0358 | Val RMSE: 1.1024
Epoch  50/150 | Train RMSE: 0.9870 | Val RMSE: 0.9459
Epoch  60/150 | Train RMSE: 0.9501 | Val RMSE: 0.8387
Epoch  70/150 | Train RMSE: 0.9128 | Val RMSE: 0.9460
Epoch  80/150 | Train RMSE: 0.8587 | Val RMSE: 0.7826
Epoch  90/150 | Train RMSE: 0.8598 | Val RMSE: 0.6029
Epoch 100/150 | Train RMSE: 0.8526 | Val RMSE: 0.6824
Epoch 110/150 | Train RMSE: 0.8008 | Val RMSE: 0.6877
Epoch 120/150 | Train RMSE: 0.8175 | Val RMSE: 0.5845
Epoch 130/150 | Train RMSE: 0.8021 | Val RMSE: 0.4978
Epoch 140/150 | Train RMSE: 0.7584 | Val RMSE: 0.5245
Epoch 150/150 | Train RMSE: 0.7282 | Val RMSE: 0.4921

✅ Meilleur Val RMSE : 0.3276


<All keys matched successfully>

In [28]:

model.eval()
# 2. On fait les prédictions sur le vrai Test Set (sans calculer les gradients)
with torch.no_grad():
    X_test_final = X_test_final.to(device)
    log_predictions = model(X_test_final).cpu().numpy()
# 3. On annule le Logarithme avec Expm1 (Exponentielle - 1) 
# pour retrouver les vrais prix (en $)
final_prices = np.expm1(log_predictions)
# 4. On crée le fichier de soumission pour Kaggle
submission = pd.DataFrame({
    'Id': test_df['Id'], # Assure-toi que tu as gardé la colonne Id du test !
    'SalePrice': final_prices.flatten()
})
submission.to_csv('submission.csv', index=False)
print("Fichier de soumission créé : submission.csv !")

Fichier de soumission créé : submission.csv !


In [29]:
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
!kaggle competitions submit -c house-prices-advanced-regression-techniques -f submission.csv -m "Mon premier MLP avec PyTorch"

100% 20.9k/20.9k [00:00<00:00, 33.2kB/s]
Successfully submitted to House Prices - Advanced Regression Techniques